# 04 - Baseline Modeling & MLflow Experimentation

> **Project:** AI-Based Interview Preparation Assistant  
> **Course:** AI252TA – Machine Learning Operations (Semester V)  
> **Status:** Phase 1 – Completed  

### Objective & Purpose
Compare MLflow-tracked models: LogisticRegression (baseline), RandomForest (compare), and final test evaluation. Build comparison tables, show confusion matrices, and analyze misclassified examples.

---


In [1]:
import json
import yaml
from pathlib import Path

import os
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import joblib

import mlflow
from mlflow.tracking import MlflowClient

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'params.yaml').exists())
params = yaml.safe_load((PROJECT_ROOT / 'params.yaml').read_text())
config = yaml.safe_load((PROJECT_ROOT / 'configs/config.yaml').read_text())

# Import custom classes so joblib can unpickle the feature pipeline
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features.build_features import TextLengthFeatures, ReferenceSimilarityFeatures  # noqa: F401

# Use absolute path for tracking URI
tracking_uri = str(PROJECT_ROOT / config['mlflow']['tracking_uri'])
mlflow.set_tracking_uri(tracking_uri)
os.environ['MLFLOW_TRACKING_URI'] = tracking_uri
client = MlflowClient(tracking_uri=tracking_uri)
experiment = client.get_experiment_by_name(config['mlflow']['experiment_name'])

print(f'Experiment: {experiment.name} (ID: {experiment.experiment_id})')

2026/10/14 22:27:31 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/likithshashidhar/MLOps/.venv/lib64/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Experiment: interview-answer-quality-baseline (ID: 123679927980488699)


## Fetch Latest MLflow Runs (FINISHED only, sorted by start_time DESC)

In [2]:
def get_latest_finished_run(run_name: str):
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=['attributes.start_time DESC'],
        max_results=10,
    )
    for run in runs:
        if run.info.status == 'FINISHED':
            return run
    return None

baseline_run = get_latest_finished_run('logistic_regression_baseline')
rf_run = get_latest_finished_run('random_forest')
final_run = get_latest_finished_run('final_test_evaluation')

print(f'LogisticRegression baseline: {baseline_run.info.run_id if baseline_run else "NOT FOUND"}')
print(f'RandomForest compare: {rf_run.info.run_id if rf_run else "NOT FOUND"}')
print(f'Final test evaluation: {final_run.info.run_id if final_run else "NOT FOUND"}')

LogisticRegression baseline: 29d2081071294bc1abf26a07ad541d5b
RandomForest compare: 284e22322f65437f89267b8d2f1fc5f8
Final test evaluation: f885b93e20b946ad8b813040df9d0ff5


## CV and Validation Metrics vs Majority Baseline

In [3]:
def extract_cv_val_metrics(run, prefix: str = ''):
    """Extract CV and validation metrics from a run."""
    m = run.data.metrics
    # For baseline (logistic_regression_baseline), cv_* and val_* are the model's metrics
    # For random_forest, same
    # For final_test_evaluation, cv_* and val_* are from both models
    return {
        'cv_f1_macro_mean': m.get(f'{prefix}cv_f1_macro_mean') or m.get('cv_f1_macro_mean_baseline'),
        'cv_f1_macro_std': m.get(f'{prefix}cv_f1_macro_std') or m.get('cv_f1_macro_std_baseline'),
        'cv_accuracy_mean': m.get(f'{prefix}cv_accuracy_mean') or m.get('cv_accuracy_mean_baseline'),
        'cv_accuracy_std': m.get(f'{prefix}cv_accuracy_std') or m.get('cv_accuracy_std_baseline'),
        'val_f1_macro': m.get(f'{prefix}val_f1_macro') or m.get('val_f1_macro_baseline'),
        'val_accuracy': m.get(f'{prefix}val_accuracy') or m.get('val_accuracy_baseline'),
    }

# For baseline and RF runs, the metrics are directly logged
def extract_simple_metrics(run):
    m = run.data.metrics
    return {
        'cv_f1_macro_mean': m.get('cv_f1_macro_mean'),
        'cv_f1_macro_std': m.get('cv_f1_macro_std'),
        'cv_accuracy_mean': m.get('cv_accuracy_mean'),
        'cv_accuracy_std': m.get('cv_accuracy_std'),
        'val_f1_macro': m.get('val_f1_macro'),
        'val_accuracy': m.get('val_accuracy'),
    }

baseline_metrics = extract_simple_metrics(baseline_run)
rf_metrics = extract_simple_metrics(rf_run)

# Majority baseline from validation (always predict class 3)
majority_val = {
    'val_f1_macro': baseline_run.data.metrics.get('val_baseline_f1_macro'),
    'val_accuracy': baseline_run.data.metrics.get('val_baseline_accuracy'),
}

print('=== CV & Validation Metrics ===')
print()
print(f'{"Metric":<25} {"LogReg (baseline)":>18} {"RandomForest":>18} {"Majority baseline (val)":>18}')
print('-' * 80)
for key, label in [
    ('cv_f1_macro_mean', 'CV f1_macro mean'),
    ('cv_f1_macro_std', 'CV f1_macro std'),
    ('cv_accuracy_mean', 'CV accuracy mean'),
    ('cv_accuracy_std', 'CV accuracy std'),
    ('val_f1_macro', 'Val f1_macro'),
    ('val_accuracy', 'Val accuracy'),
]:
    b = baseline_metrics.get(key)
    r = rf_metrics.get(key)
    mb = majority_val.get(key.replace('val_', 'val_baseline_')) if key.startswith('val_') else None
    b_str = f'{b:.4f}' if b is not None else 'N/A'
    r_str = f'{r:.4f}' if r is not None else 'N/A'
    mb_str = f'{mb:.4f}' if mb is not None else 'N/A'
    print(f'{label:<25} {b_str:>18} {r_str:>18} {mb_str:>18}')

=== CV & Validation Metrics ===

Metric                     LogReg (baseline)       RandomForest Majority baseline (val)
--------------------------------------------------------------------------------
CV f1_macro mean                      0.3717             0.3690                N/A
CV f1_macro std                       0.0456             0.0434                N/A
CV accuracy mean                      0.5358             0.5670                N/A
CV accuracy std                       0.0118             0.0256                N/A
Val f1_macro                          0.3418             0.3778                N/A
Val accuracy                          0.5153             0.5764                N/A


> **Takeaway:** LogisticRegression and RandomForest have nearly identical CV f1_macro (~0.37). RandomForest wins on validation f1_macro (0.3778 vs 0.3418) and validation accuracy (0.5764 vs 0.5153). The majority baseline (always predict class 3) achieves higher validation accuracy (0.6550) but very low f1_macro (0.1979), confirming class imbalance makes accuracy a poor metric.

## Test Metrics (from reports/metrics.json)

In [4]:
# Load test metrics from reports/metrics.json
metrics_path = PROJECT_ROOT / 'reports/metrics.json'
with open(metrics_path) as f:
    test_metrics = json.load(f)

print('=== Test Metrics (from reports/metrics.json) ===')
print()

lr_test = test_metrics['test_evaluation']['logistic_regression']['test_metrics']
rf_test = test_metrics['test_evaluation']['random_forest']['test_metrics']
majority_test = test_metrics['test_evaluation']['majority_class_baseline']

print(f'{"Metric":<25} {"LogReg":>12} {"RandomForest":>14} {"Majority Baseline":>18}')
print('-' * 70)
for key, label in [
    ('accuracy', 'Accuracy'),
    ('precision_weighted', 'Precision (weighted)'),
    ('recall_weighted', 'Recall (weighted)'),
    ('f1_weighted', 'F1 (weighted)'),
    ('f1_macro', 'F1 (macro)'),
]:
    lr = lr_test.get(key)
    rf = rf_test.get(key)
    mb = majority_test.get(key)
    lr_str = f'{lr:.4f}' if lr is not None else 'N/A'
    rf_str = f'{rf:.4f}' if rf is not None else 'N/A'
    mb_str = f'{mb:.4f}' if mb is not None else 'N/A'
    print(f'{label:<25} {lr_str:>12} {rf_str:>14} {mb_str:>18}')

=== Test Metrics (from reports/metrics.json) ===

Metric                          LogReg   RandomForest  Majority Baseline
----------------------------------------------------------------------
Accuracy                        0.5611         0.5901             0.5963
Precision (weighted)            0.5591         0.5333                N/A
Recall (weighted)               0.5611         0.5901                N/A
F1 (weighted)                   0.5577         0.5544                N/A
F1 (macro)                      0.3669         0.3925             0.1868


In [5]:
# Generate test takeaway from metrics.json
metrics_path = PROJECT_ROOT / 'reports/metrics.json'
with open(metrics_path) as f:
    test_metrics = json.load(f)

lr_test = test_metrics['test_evaluation']['logistic_regression']['test_metrics']
rf_test = test_metrics['test_evaluation']['random_forest']['test_metrics']
majority_test = test_metrics['test_evaluation']['majority_class_baseline']

rf_beats = []
lr_beats = []
for key, label in [
    ('accuracy', 'Accuracy'),
    ('precision_weighted', 'Precision (weighted)'),
    ('recall_weighted', 'Recall (weighted)'),
    ('f1_weighted', 'F1 (weighted)'),
    ('f1_macro', 'F1 (macro)'),
]:
    lr = lr_test.get(key)
    rf = rf_test.get(key)
    if rf is not None and lr is not None:
        if rf > lr:
            rf_beats.append(label)
        elif lr > rf:
            lr_beats.append(label)

rf_beat_str = ', '.join(rf_beats) if rf_beats else 'none'
lr_beat_str = ', '.join(lr_beats) if lr_beats else 'none'
print(f'RandomForest beats LogisticRegression on: {rf_beat_str}')
print(f'LogisticRegression beats RandomForest on: {lr_beat_str}')

print(f'Both beat majority baseline on F1 macro: LR {lr_test["f1_macro"]:.4f} vs majority {majority_test["f1_macro"]:.4f}, RF {rf_test["f1_macro"]:.4f} vs majority {majority_test["f1_macro"]:.4f}')
print(f'Neither beats majority on accuracy: LR {lr_test["accuracy"]:.4f}, RF {rf_test["accuracy"]:.4f} vs majority {majority_test["accuracy"]:.4f}')
print(f'> **Takeaway:** On test data, RandomForest outperforms LogisticRegression on {rf_beat_str} while LogisticRegression outperforms on {lr_beat_str}. Both beat the majority baseline on F1 macro ({lr_test["f1_macro"]:.4f} and {rf_test["f1_macro"]:.4f} vs {majority_test["f1_macro"]:.4f}) but neither beats it on accuracy ({lr_test["accuracy"]:.4f} and {rf_test["accuracy"]:.4f} vs {majority_test["accuracy"]:.4f}).')


RandomForest beats LogisticRegression on: Accuracy, Recall (weighted), F1 (macro)
LogisticRegression beats RandomForest on: Precision (weighted), F1 (weighted)
Both beat majority baseline on F1 macro: LR 0.3669 vs majority 0.1868, RF 0.3925 vs majority 0.1868
Neither beats majority on accuracy: LR 0.5611, RF 0.5901 vs majority 0.5963
> **Takeaway:** On test data, RandomForest outperforms LogisticRegression on Accuracy, Recall (weighted), F1 (macro) while LogisticRegression outperforms on Precision (weighted), F1 (weighted). Both beat the majority baseline on F1 macro (0.3669 and 0.3925 vs 0.1868) but neither beats it on accuracy (0.5611 and 0.5901 vs 0.5963).


> **Note:** Confusion matrices are recomputed from the saved models on the test split for display only; metrics match reports/metrics.json; no model was changed or tuned.

## Primary Model Confusion Matrix (from final_test_evaluation run)

In [6]:
# Load confusion matrix from final_test_evaluation run artifacts
import tempfile
import os

def load_confusion_matrix_from_run(run_id, artifact_name):
    """Load confusion matrix image from MLflow artifacts."""
    with tempfile.TemporaryDirectory() as tmpdir:
        local_path = client.download_artifacts(run_id, artifact_name, tmpdir)
        # For images, we can't easily display in notebook without PIL
        return local_path

# Instead, recompute confusion matrices from test data (we have the models)
test_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/test.parquet')
X_test = test_df.drop(columns=[params['data']['target_column']])
y_test = test_df[params['data']['target_column']].values

# Load feature pipeline
feature_pipeline = joblib.load(PROJECT_ROOT / 'models/feature_pipeline.joblib')
X_test_feat = feature_pipeline.transform(X_test)

# Load models
lr_model = joblib.load(PROJECT_ROOT / 'models/baseline_model.joblib')
rf_model = joblib.load(PROJECT_ROOT / 'models/rf_model.joblib')

y_lr_pred = lr_model.predict(X_test_feat)
y_rf_pred = rf_model.predict(X_test_feat)

from sklearn.metrics import confusion_matrix

target_classes = params['data']['target_classes']
class_names = [target_classes[i] for i in range(4)]

cm_lr = confusion_matrix(y_test, y_lr_pred)
cm_rf = confusion_matrix(y_test, y_rf_pred)

print('=== LogisticRegression Confusion Matrix (Test) ===')
print(pd.DataFrame(cm_lr, index=class_names, columns=class_names).to_string())
print()
print('=== RandomForest Confusion Matrix (Test) ===')
print(pd.DataFrame(cm_rf, index=class_names, columns=class_names).to_string())

=== LogisticRegression Confusion Matrix (Test) ===
                   Poor  Needs Improvement  Good  Excellent
Poor                  3                  4     2          1
Needs Improvement     7                 31    26         35
Good                  8                 18    16         44
Excellent             7                 23    37        221

=== RandomForest Confusion Matrix (Test) ===
                   Poor  Needs Improvement  Good  Excellent
Poor                  3                  3     2          2
Needs Improvement     2                 33    16         48
Good                  1                 20     7         58
Excellent             2                 22    22        242


In [7]:
# Assert recomputed test metrics match reports/metrics.json
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

metrics_path = PROJECT_ROOT / 'reports/metrics.json'
with open(metrics_path) as f:
    test_metrics = json.load(f)

lr_test_acc = test_metrics['test_evaluation']['logistic_regression']['test_metrics']['accuracy']
lr_test_f1 = test_metrics['test_evaluation']['logistic_regression']['test_metrics']['f1_macro']
rf_test_acc = test_metrics['test_evaluation']['random_forest']['test_metrics']['accuracy']
rf_test_f1 = test_metrics['test_evaluation']['random_forest']['test_metrics']['f1_macro']

lr_acc = accuracy_score(y_test, y_lr_pred)
lr_f1 = f1_score(y_test, y_lr_pred, average='macro')
rf_acc = accuracy_score(y_test, y_rf_pred)
rf_f1 = f1_score(y_test, y_rf_pred, average='macro')

assert np.isclose(lr_acc, lr_test_acc), f'LR accuracy mismatch: {lr_acc:.6f} vs {lr_test_acc:.6f}'
assert np.isclose(lr_f1, lr_test_f1), f'LR f1_macro mismatch: {lr_f1:.6f} vs {lr_test_f1:.6f}'
assert np.isclose(rf_acc, rf_test_acc), f'RF accuracy mismatch: {rf_acc:.6f} vs {rf_test_acc:.6f}'
assert np.isclose(rf_f1, rf_test_f1), f'RF f1_macro mismatch: {rf_f1:.6f} vs {rf_test_f1:.6f}'

print('All assertions passed: recomputed test metrics match reports/metrics.json')
print(f'LR accuracy: {lr_acc:.6f} == {lr_test_acc:.6f}')
print(f'LR f1_macro: {lr_f1:.6f} == {lr_test_f1:.6f}')
print(f'RF accuracy: {rf_acc:.6f} == {rf_test_acc:.6f}')
print(f'RF f1_macro: {rf_f1:.6f} == {rf_test_f1:.6f}')


All assertions passed: recomputed test metrics match reports/metrics.json
LR accuracy: 0.561077 == 0.561077
LR f1_macro: 0.366939 == 0.366939
RF accuracy: 0.590062 == 0.590062
RF f1_macro: 0.392525 == 0.392525


In [8]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('LogisticRegression - Test Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('RandomForest (Primary) - Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

/tmp/ipykernel_68583/3612417560.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** RandomForest (primary model) shows better diagonal concentration, especially for 'Excellent' and 'Needs Improvement' classes. Both models struggle with 'Good' class (class 2), often confusing it with 'Excellent' or 'Needs Improvement'. The 'Poor' class has very few test samples (10), making its metrics unstable.

## 10 Misclassified TRAIN-or-VAL Examples with Commentary

In [9]:
# Load train and val data
train_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/train.parquet')
val_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/val.parquet')

# Transform with feature pipeline
X_train = train_df.drop(columns=[params['data']['target_column']])
y_train = train_df[params['data']['target_column']].values
X_val = val_df.drop(columns=[params['data']['target_column']])
y_val = val_df[params['data']['target_column']].values

X_train_feat = feature_pipeline.transform(X_train)
X_val_feat = feature_pipeline.transform(X_val)

# Get predictions from primary model (RandomForest)
y_train_pred = rf_model.predict(X_train_feat)
y_val_pred = rf_model.predict(X_val_feat)

# Find misclassified examples
train_misclassified = np.where(y_train_pred != y_train)[0]
val_misclassified = np.where(y_val_pred != y_val)[0]

print(f'Train misclassified: {len(train_misclassified)} / {len(y_train)}')
print(f'Val misclassified: {len(val_misclassified)} / {len(y_val)}')

Train misclassified: 5 / 1561
Val misclassified: 97 / 229


In [10]:
# Show 10 misclassified examples (5 from train, 5 from val)
np.random.seed(42)
train_sample = np.random.choice(train_misclassified, min(5, len(train_misclassified)), replace=False)
val_sample = np.random.choice(val_misclassified, min(5, len(val_misclassified)), replace=False)

def show_example(df, idx, y_true, y_pred, split_name):
    row = df.iloc[idx]
    print(f'=== {split_name} Example {idx} ===')
    print(f'Question ID: {row["question_id"]}')
    print(f'Question: {row["question"][:300]}...')
    print(f'Reference: {row["reference_answer"][:200]}...')
    print(f'Candidate: {row["candidate_answer"][:300]}...')
    print(f'True: {target_classes[y_true[idx]]} ({y_true[idx]})')
    print(f'Pred: {target_classes[y_pred[idx]]} ({y_pred[idx]})')
    print(f'score_avg: {row["score_avg"]:.2f}')
    print()

print('--- TRAIN MISCLASSIFIED ---')
for idx in train_sample:
    show_example(train_df, idx, y_train, y_train_pred, 'Train')

print('--- VAL MISCLASSIFIED ---')
for idx in val_sample:
    show_example(val_df, idx, y_val, y_val_pred, 'Val')

--- TRAIN MISCLASSIFIED ---
=== Train Example 721 ===
Question ID: E07.Q04
Question: How are linked lists passed as arguments to a function? <STOP>...
Reference: By reference. <STOP>...
Candidate: They are passed by reference because you want the function to change the pointer <STOP>...
True: Excellent (3)
Pred: Poor (0)
score_avg: 4.50

=== Train Example 1173 ===
Question ID: E10.Q07
Question: How many comparisons does it take to find an element in a binary search tree? <STOP>...
Reference: The height of the tree -LRB- or log of the number of elements in the tree -RRB- . <STOP>...
Candidate: logn <STOP>...
True: Excellent (3)
Pred: Needs Improvement (1)
score_avg: 5.00

=== Train Example 916 ===
Question ID: E09.Q01
Question: What is a queue? <STOP>...
Reference: A data structure that can store elements, which has the property that the last item added will be the last to be removed -LRB- or first-in-first-out -RRB- . <STOP>...
Candidate: Like a linked list but first in first out <STOP

> **Commentary on Misclassified Examples:**

1. **Borderline scores**: Many misclassified examples have score_avg near class boundaries (e.g., 3.4-3.6 for class 1/2 boundary, 4.4-4.6 for class 2/3). The model's prediction often matches the adjacent class.

2. **Short but correct answers**: Some 'Good' or 'Excellent' answers are concise but technically correct. The model may penalize brevity due to correlation between length and quality.

3. **Verbose but shallow answers**: Some 'Needs Improvement' answers are long but lack key concepts. The model may over-reward length.

4. **Reference similarity mismatch**: Examples where candidate answer is semantically similar to reference but phrased differently can confuse the cosine similarity feature.

5. **Class imbalance effect**: The model tends to predict 'Excellent' (majority class) for uncertain cases, leading to false positives on 'Good' and 'Needs Improvement' examples.

These patterns suggest the model captures surface-level correlates (length, keyword overlap) but struggles with deep semantic understanding—expected for a TF-IDF + LR/RF baseline.

## MLflow Metric Naming Clarification

In [11]:
# Show what metrics are logged in each run
print('=== MLflow Metric Names ===')
print()

for name, run in [('LogisticRegression baseline', baseline_run),
                 ('RandomForest', rf_run),
                 ('Final test evaluation', final_run)]:
    print(f'--- {name} ({run.info.run_id[:8]}) ---')
    metrics = sorted(run.data.metrics.keys())
    for m in metrics:
        print(f'  {m}: {run.data.metrics[m]:.4f}')
    print()

=== MLflow Metric Names ===

--- LogisticRegression baseline (29d20810) ---
  cv_accuracy_mean: 0.5358
  cv_accuracy_std: 0.0118
  cv_f1_macro_mean: 0.3717
  cv_f1_macro_std: 0.0456
  cv_fold_0_accuracy: 0.5321
  cv_fold_0_f1_macro: 0.4273
  cv_fold_1_accuracy: 0.5548
  cv_fold_1_f1_macro: 0.2952
  cv_fold_2_accuracy: 0.5241
  cv_fold_2_f1_macro: 0.3706
  cv_fold_3_accuracy: 0.5434
  cv_fold_3_f1_macro: 0.3580
  cv_fold_4_accuracy: 0.5246
  cv_fold_4_f1_macro: 0.4074
  train_accuracy: 0.7918
  train_f1_macro: 0.7500
  train_f1_weighted: 0.7996
  train_precision_weighted: 0.8368
  train_recall_weighted: 0.7918
  val_accuracy: 0.5153
  val_baseline_accuracy: 0.6550
  val_baseline_f1_macro: 0.1979
  val_baseline_f1_weighted: 0.5185
  val_baseline_precision_weighted: 0.4291
  val_baseline_recall_weighted: 0.6550
  val_f1_macro: 0.3418
  val_f1_weighted: 0.5441
  val_precision_weighted: 0.5900
  val_recall_weighted: 0.5153

--- RandomForest (284e2232) ---
  cv_accuracy_mean: 0.5670
  cv_acc

> **Note on Metric Naming:**

- In `logistic_regression_baseline` and `random_forest` runs: `cv_*`, `val_*`, `train_*` refer to **that model's** metrics.
- In `final_test_evaluation` run: `cv_f1_macro_mean_baseline`, `val_f1_macro_baseline` refer to **LogisticRegression** (the baseline model); `cv_f1_macro_mean_rf`, `val_f1_macro_rf` refer to **RandomForest**; `test_baseline_*` refers to the **majority-class baseline**.
- Test metrics in `reports/metrics.json` use clear model-specific keys: `logistic_regression`, `random_forest`, `majority_class_baseline`.

Tables in this notebook explicitly label each metric by model to avoid confusion.